# **Load Results**

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

INPUT_FILE  = 'Gamage_Recruiters_Sentiment_Results.xlsx'
OUTPUT_FILE = 'Gamage_Recruiters_Aggregated_Results.xlsx'

df = pd.read_excel(INPUT_FILE, sheet_name='Sentiment_Results')
df['date'] = pd.to_datetime(df['date'], errors='coerce')

print(f'Rows loaded: {len(df)}')
df[['mention_id', 'comment_text', 'final_sentiment']].head(5)

Rows loaded: 21


,mention_id,comment_text,final_sentiment
0,LI-087,What is the industry,Neutral
1,LI-090,"Exciting opportunity, Highly recommend, Sharin...",Positive
2,LI-091,"Great opportunity, Amazing opportunity",Positive
3,LI-092,"Thanks for sharing, Poornima, Thanks for shari...",Positive
4,LI-093,"I highly recommend, Great opportunity, Amazing...",Positive


# **Overall Sentiment Counts and Percentages**

In [2]:
total            = len(df)
sentiment_counts = df['final_sentiment'].value_counts()
sentiment_pct    = (sentiment_counts / total * 100).round(1)

summary = pd.DataFrame({
    'Count'      : sentiment_counts,
    'Percentage' : sentiment_pct
})

print('=' * 40)
print('OVERALL SENTIMENT SUMMARY')
print('=' * 40)
print(summary)
print(f'{"─" * 40}')
print(f'Total comments analyzed: {total}')
print()
for label, row in summary.iterrows():
    bar = '█' * int(row['Percentage'] / 2)
    print(f'  {label:<10} {row["Percentage"]:>5}%  {bar}')

OVERALL SENTIMENT SUMMARY
                 Count  Percentage
final_sentiment                   
Positive            19        90.5
Neutral              2         9.5
────────────────────────────────────────
Total comments analyzed: 21

  Positive    90.5%  █████████████████████████████████████████████
  Neutral      9.5%  ████


# **Sentiment by Post Type**

In [3]:
by_type = df.groupby(
    ['post_type', 'final_sentiment']
).size().unstack(fill_value=0)

by_type['Total'] = by_type.sum(axis=1)

by_type_pct = by_type.drop(columns='Total')\
    .div(by_type['Total'], axis=0).mul(100).round(1)

print('Counts:')
print(by_type)
print()
print('Percentages:')
print(by_type_pct)

Counts:
final_sentiment  Neutral  Positive  Total
post_type                                
Job Post               2        14     16
Marketing Post         0         5      5

Percentages:
final_sentiment  Neutral  Positive
post_type                         
Job Post            12.5      87.5
Marketing Post       0.0     100.0


# **Meaningful vs Low Quality Breakdown**

In [4]:
print('ALL COMMENTS — Sentiment vs Quality Flag')
print('=' * 55)
quality_breakdown = df.groupby(
    ['is_meaningful', 'final_sentiment']
).size().unstack(fill_value=0)

quality_breakdown.index = ['Low Quality', 'Meaningful']
print(quality_breakdown)
print()
print('Meaningful comments only:')
meaningful = df[df['is_meaningful'] == True]
print(meaningful['final_sentiment'].value_counts())

ALL COMMENTS — Sentiment vs Quality Flag
final_sentiment  Neutral  Positive
Low Quality            1         1
Meaningful             1        18

Meaningful comments only:
final_sentiment
Positive    18
Neutral      1
Name: count, dtype: int64


# **Engagement vs Sentiment**

In [6]:
engagement = df.groupby('final_sentiment')[
    ['likes', 'shares', 'num_comments']
].mean().round(2)

print('AVERAGE ENGAGEMENT BY SENTIMENT')
print('=' * 45)
print(engagement)
print()

AVERAGE ENGAGEMENT BY SENTIMENT
                 likes  shares  num_comments
final_sentiment                             
Neutral          12.00    4.00          1.00
Positive         10.68    4.32          2.58



# **Save All Aggregated Results**

In [8]:
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    summary.reset_index().to_excel(
        writer, sheet_name='Overall_Summary', index=False)
    by_type.reset_index().to_excel(
        writer, sheet_name='By_Post_Type', index=False)
    engagement.reset_index().to_excel(
        writer, sheet_name='Engagement_Metrics', index=False)
    df.to_excel(
        writer, sheet_name='Full_Data', index=False)

print('STEP 6 COMPLETE')


STEP 6 COMPLETE
